# Comparing Kubernetes Workload Types

> Notes to come — placeholder for comparing Kubernetes workload types: Deployments, StatefulSets, DaemonSets, and Jobs.

## Purpose

I wanted to understand which Kubernetes workload type to reach for in each situation. The cluster has several controllers that all look like YAML with a `kind`, and the difference between a Deployment and a StatefulSet is not obvious from a distance. This notebook compares Pods, Deployments, StatefulSets, DaemonSets, and Jobs with interactive examples so I can see the trade-offs in action.

## Setup

Make sure `kubectl` is installed and pointed at a cluster (or minikube). The examples below are mostly read-only inspections, but the apply steps will create real objects — run them in a scratch namespace you don't care about.

```bash
kubectl version --short 2>/dev/null || kubectl version
kubectl config current-context
```

In [ ]:
# Scratch namespace for the examples
NS="workload-demo"
kubectl create namespace $NS -o yaml | kubectl apply -f -

## What is a Pod?

A Pod is the smallest unit Kubernetes manages. It can hold one or more containers that share a network namespace and storage volumes. Everything else in Kubernetes wraps Pods — a Deployment does not run containers, it runs Pods.

In [ ]:
# A bare Pod — the raw building block
cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: bare-pod
  namespace: workload-demo
spec:
  containers:
  - name: web
    image: nginx:1.27
EOF
kubectl get pod bare-pod -n workload-demo -o wide

## Deployment — stateless, self-healing replicas

A Deployment manages a set of identical Pods. Its job is to keep N replicas running, roll out new versions, and roll back when something goes wrong. Pods are disposable: if one dies, the Deployment replaces it with a fresh one, and nobody cares which Pod served which request.

**When to use:** stateless web services, APIs, workers where any replica can serve any request.

In [ ]:
cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web-deploy
  namespace: workload-demo
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
      - name: web
        image: nginx:1.27
        ports:
        - containerPort: 80
EOF
kubectl rollout status deploy/web-deploy -n workload-demo
kubectl get pods -n workload-demo -l app=web -o wide

## StatefulSet — stable identity and ordered pods

A StatefulSet gives each Pod a stable name and identity (`web-0`, `web-1`, `web-2`), creates them in order, and deletes them in reverse order. It also provisions a stable volume for each Pod from a PersistentVolumeClaim template, so `web-0` keeps its data even if it moves to another node.

**When to use:** databases, message queues, distributed stores — anything where a Pod's identity or its disk matters.

```bash
# Delete the Deployment first — StatefulSet pods need stable names
kubectl delete deploy web-deploy -n workload-demo --wait

In [ ]:
cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: web-sts
  namespace: workload-demo
spec:
  serviceName: web-svc
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
      - name: web
        image: nginx:1.27
        ports:
        - containerPort: 80
        name: http
  volumeClaimTemplates:
  - metadata:
      name: www
    spec:
      accessModes: ["ReadWriteOnce"]
      resources:
        requests:
          storage: 1Gi
EOF
kubectl get pods -n workload-demo -l app=web
kubectl get pvc -n workload-demo

## DaemonSet — one Pod per node

A DaemonSet ensures every node (or a subset selected by a node selector) runs exactly one copy of a Pod. If you add a node, the DaemonSet schedules a Pod on it. If you remove a node, the Pod is gone. There are no guaranteed replicas — the count is always `len(nodes)`.

**When to use:** node-level agents: log collectors, metrics exporters, CNI plugins, host-level daemons.

In [ ]:
cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: node-agent
  namespace: workload-demo
spec:
  selector:
    matchLabels:
      app: node-agent
  template:
    metadata:
      labels:
        app: node-agent
    spec:
      hostNetwork: true
      tolerations:
      - key: node-role.kubernetes.io/control-plane
        effect: NoSchedule
      containers:
      - name: agent
        image: busybox:1.36
        command: ["sh", "-c", "while true; do echo agent-running; sleep 30; done"]
EOF
kubectl get daemonset node-agent -n workload-demo
kubectl get pods -n workload-demo -l app=node-agent -o wide

## Job — run something to completion

A Job creates one or more Pods and keeps retrying until the work finishes successfully. Once the required number of Pods succeed, the Job is marked complete and no new Pods are started. A CronJob wraps a Job with a schedule.

**When to use:** one-off workloads, batch processing, migrations, anything with a defined end.

In [ ]:
cat <<'EOF' | kubectl apply -f -
apiVersion: batch/v1
kind: Job
metadata:
  name: pi-job
  namespace: workload-demo
spec:
  backoffLimit: 4
  template:
    spec:
      restartPolicy: OnFailure
      containers:
      - name: pi
        image: perl:5.36
        command: ["perl", "-e", "print 3.14"]
EOF
kubectl wait --for=condition=complete job/pi-job -n workload-demo --timeout=120s
kubectl get job pi-job -n workload-demo

## Side-by-side comparison

| Workload | Pod identity | Stable storage | Per-node placement | Ordering | Typical use |
|---|---|---|---|---|---|
| Pod | manual, fixed name | manual volume | none | none | building block only |
| Deployment | random, disposable | none | none | none | stateless web services |
| StatefulSet | stable (`web-0`…`web-N`) | PVC per pod | none | ordered create/delete | databases, queues |
| DaemonSet | per-node, disposable | none | one per node | none | node agents |
| Job | disposable | none | none | runs to completion | batch, migrations |

## How I'd pick

- Need to scale a stateless service and roll out updates? **Deployment**
- Need stable network names and persistent disks per instance? **StatefulSet**
- Need something on every node, always? **DaemonSet**
- Need something that finishes and exits? **Job**
- Need that same thing on a schedule? **CronJob**

## Verify

```bash
kubectl get deployment,statefulset,daemonset,job -n workload-demo
kubectl logs -n workload-demo job/pi-job
```

## What I'd try next

I'd compare these workload types against Helm and Kustomize overlays to see how the same Deployment looks when generated by a template, and then walk through `kubectl rollout restart/undo` to feel the upgrade and rollback paths for real.
